In [ ]:
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import numpy as np
import pandas as pd

import gzip
import re

ROOT = Path(os.path.abspath(".")).parent

In [ ]:
exp_name = "250716_atac_finetune"
chk = "best_valid_loss"
BASE = f"{ROOT}/Res/{exp_name}/analysis_{chk}/raw_data/var_eff/"

label_meta = pd.read_csv(f"{ROOT}/Data/lc_HM_ATAC_v1/label_meta.csv", index_col=1)
vcf = pd.read_csv(f"{ROOT}/Data/source/gwas.catlog.vcf", sep="\t")

In [ ]:
vcf

# Functions

In [ ]:
def search_anno(target_gene, target_transcript=None):

    gtf_path = f"{ROOT}/Data/source/gencode.v48.annotation.gtf.gz"

    exons = []
    with gzip.open(gtf_path, "rt") as fp:
        for line in fp:
            if line.startswith("#"):
                continue
            cols = line.rstrip("\n").split("\t")
            if cols[2] != "exon":
                continue
            attr_str = cols[8]
            if f'gene_name "{target_gene}"' not in attr_str:
                continue

            # 解析 attributes
            attrs = dict(re.findall(r'(\S+) "([^"]+)"', attr_str))
            tx_id = attrs.get("transcript_id")
            if target_transcript and tx_id != target_transcript:
                continue

            start, end = int(cols[3]), int(cols[4])
            exons.append((start, end))

    exons.sort(key=lambda x: x[0])

    annos = {}
    for i, (start, end) in enumerate(exons, start=1):
        key = target_gene if i == 1 else f"{target_gene}.{i}"
        annos[key] = {"idx": (start, end), "show_text": False}

    return annos

In [ ]:
def prepare_data(track, var_idx, annos):

    track_dim = label_meta.loc[track, "dim"]
    chr_name, pos, ref, alt = vcf.iloc[var_idx, [0, 1, 3, 4]]

    annos.update({"Mut": {"idx": (pos, pos), "show_text": True}})

    with h5py.File(f"{BASE}/{chr_name}_{ref}{pos}{alt}.h5", "r") as f:

        data = {
            "label": f["data/label"][:, track_dim],
            "pred_wt": f["data/pred_wt"][:, track_dim],
            "pred_alt": f["data/pred_alt"][:, track_dim],
            "diff": f["data/diff"][:, track_dim],
        }

        metadata = {
            "context_start": f.attrs["context_start"],
            "context_end": f.attrs["context_end"],
        }

    # add basic plot track data
    plot_data = list(data.values())
    plot_title = ["Target", "Wt Pred", "Mut Pred", "Diff (Raw)"]

    trim = (524288 // 32 - data["diff"].shape[0]) // 2
    x = np.arange(metadata["context_start"], metadata["context_end"]).reshape(-1, 32)[trim:-trim]
    for k, v in annos.items():
        start, end = v["idx"]
        bin_start = np.where(x == start)[0]
        bin_end = np.where(x == end)[0]
        v["bin"] = (bin_start, bin_end)

    return data, plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track

In [ ]:
def plot(plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track):

    # start to plot
    n = len(plot_data) + 1
    height_ratios = [1] * len(plot_data) + [0.7]
    fig, axes = plt.subplots(
        nrows=n, ncols=1, figsize=(8, n * 1.5), sharex=True, gridspec_kw={"height_ratios": height_ratios}
    )

    x = x[:, 0]

    # add tracks
    for i, ax in enumerate(axes[:-1]):
        ax.plot(x, plot_data[i])
        ax.set_title(plot_title[i])
        ax.set_ylabel(None)
        ax.set_xlabel(None)

    # add annos
    ax = axes[-1]
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_ylabel(None)
    ax.set_xlabel(None)

    linewidth = 1.5
    rec_width = 0.2
    ax.hlines(0.5, x.min(), x.max(), color="black", linewidth=linewidth)
    ax.set_title(f"Chromosome {chr_name[3:]}, {ref} {pos} {alt}, {track}")

    for i, (k, v) in enumerate(annos.items()):
        # start‐ and end‐positions as scalars
        x0 = x[v["bin"][0]].item()
        width = (x[v["bin"][1]] - x[v["bin"][0]]).item()

        # draw the rectangle
        rect = Rectangle(
            (x0, 0.5 - 0.5 * rec_width),
            width,
            rec_width,
            facecolor="lightblue",
            edgecolor="black",
            linewidth=linewidth,
        )
        ax.add_patch(rect)

        # add the label 'k' centered on top of the rectangle
        if v["show_text"]:
            ax.text(
                x0 + width / 2,  # x‐position: middle of the rect
                0.5 + rec_width / 2 + 0.04,  # y‐position: just above the rect
                k,  # the annotation text
                ha="center",  # horizontal alignment
                va="bottom",  # vertical alignment
                fontsize="large",
                rotation=0,
            )

    plt.tight_layout()
    plt.show()

# Case

In [ ]:
annos = {}

In [ ]:
target_gene = "CACNA1C"
target_transcript = "ENST00000682544.1"
annos.update(search_anno(target_gene, target_transcript))

In [ ]:
annos["CACNA1C"]["show_text"] = True

In [ ]:
annos

Case

In [ ]:
track = "STR-D1-MSN-GABA_K27Ac"

data, plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track = prepare_data(
    track=track, var_idx=7, annos=annos
)

to_remove = []
for bin in annos.keys():
    if bin.startswith("CACNA1C"):
        if len(annos[bin]['bin'][0]) == 0:
            print(bin)
            to_remove.append(bin)
for bin in to_remove:
    annos.pop(bin)
    
plot(plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track)

Control

In [ ]:
track = "Microglia_K27Ac"

data, plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track = prepare_data(
    track=track, var_idx=4, annos=annos
)

plot(plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track)